<center><font size=6>MCP World Explorer</center></font>

# 1. What is MCP (Model Context Protocol)?

## The Problem

LLMs are powerful reasoners, but they're locked inside a text bubble. They can't check the weather, look up country facts, or access any live data on their own. Traditionally, every integration requires custom glue code on the client side — one adapter for weather, another for country data, another for holidays, and so on.

## The Solution: MCP

**Model Context Protocol (MCP)** is an open standard that gives LLMs a universal way to connect to external data and tools. Think of it as a **USB-C port for AI** — one standardized interface that any tool can plug into.

### Architecture

```
┌─────────────┐       MCP        ┌─────────────────┐      HTTP       ┌──────────────┐
│  LLM Agent  │ ◄──────────────► │   MCP Server    │ ◄─────────────► │  Real APIs   │
│  (Client)   │   stdio/SSE      │  (server.py)    │                 │  (Internet)  │
└─────────────┘                   └─────────────────┘                 └──────────────┘
```

### The Three Primitives

MCP defines three types of capabilities a server can expose:

| Primitive | Purpose | Who Triggers | Example |
|-----------|---------|--------------|----------|
| **Tools** | Actions the LLM can invoke | The LLM decides | `get_weather("Tokyo")` |
| **Resources** | Read-only reference data | The client/user | `data://country-codes` |
| **Prompts** | Reusable prompt templates | The client/user | `travel_brief("Japan")` |

In this notebook, we'll build an MCP server that wraps **three free public APIs** (no API keys needed!) and then connect an LLM agent that uses these tools to research travel destinations.

# 2. Setup

## Install Dependencies

In [ ]:
!pip install langchain==0.3.27 \
            langchain-openai==0.3.0 \
            langchain-mcp-adapters==0.1.14 \
            langgraph==1.0.1

**Note:**
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells starting from the next cell onward.

## Imports

In [ ]:
from typing import Optional
import json
import os
import asyncio
import subprocess
import sys
from langchain_core.messages import HumanMessage
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_openai import ChatOpenAI
from langgraph.prebuilt.chat_agent_executor import create_react_agent

from pathlib import Path
from getpass import getpass


def load_openai_config(required_for: str) -> dict:
    config_path = Path("config.json")
    config = {}
    if config_path.exists():
        config = json.loads(config_path.read_text())

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    def read_value(key: str, default: str | None = None) -> str | None:
        if config.get(key):
            return config[key]
        env_value = os.environ.get(key)
        if env_value:
            return env_value
        if userdata is not None:
            try:
                secret_value = userdata.get(key)
            except Exception:
                secret_value = None
            if secret_value:
                return secret_value
        return default

    openai_api_key = read_value("OPENAI_API_KEY")
    openai_api_base = read_value("OPENAI_API_BASE", "https://api.openai.com/v1")

    if not openai_api_key:
        openai_api_key = getpass("Enter your OPENAI_API_KEY: ").strip()

    if not openai_api_key:
        raise ValueError(f"OPENAI_API_KEY is required to run {required_for}.")

    config = {
        "OPENAI_API_KEY": openai_api_key,
        "OPENAI_API_BASE": openai_api_base,
    }
    config_path.write_text(json.dumps(config, indent=2))

    print(f"Saved OpenAI config to {config_path.resolve()}")
    print(f"Using OPENAI_API_BASE={openai_api_base}")
    return config



In [ ]:
import nest_asyncio
nest_asyncio.apply()

# 3. Building the MCP Server

Our server wraps three free public APIs:

| API | Base URL | What it provides |
|-----|----------|------------------|
| **Open-Meteo Geocoding** | `geocoding-api.open-meteo.com` | City name to latitude/longitude |
| **Open-Meteo Weather** | `api.open-meteo.com` | Current weather + 7-day forecast |
| **REST Countries** | `restcountries.com/v3.1` | Country details (population, currency, languages) |
| **Nager.Date** | `date.nager.at/api/v3` | Public holidays by country and year |

The cell below writes `server.py` to disk. When the MCP client starts the server as a subprocess, this file is what runs.

In [ ]:
%%writefile server.py
# ============================================================
# MCP World Explorer Server
# Exposes travel research tools via Model Context Protocol
# ============================================================

from mcp.server.fastmcp import FastMCP
import httpx

# Create the MCP server instance
mcp = FastMCP("WorldExplorer")


# ── Private helper ───────────────────────────────────────────
# Shared geocoding function used by weather tools.
# Converts a city name to (latitude, longitude) via Open-Meteo.

async def _geocode(city: str) -> dict:
    """Look up latitude/longitude for a city name."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1}
        )
        data = resp.json()
        if "results" not in data:
            return None
        r = data["results"][0]
        return {
            "name": r["name"],
            "country": r.get("country", "Unknown"),
            "lat": r["latitude"],
            "lon": r["longitude"]
        }


# ── Tool 1: Get Weather ──────────────────────────────────────
# Geocodes the city, then fetches current conditions + 7-day
# forecast from the Open-Meteo API.

@mcp.tool()
async def get_weather(city: str) -> str:
    """Get current weather and 7-day forecast for a city."""
    geo = await _geocode(city)
    if not geo:
        return f"Could not find city: {city}"

    async with httpx.AsyncClient() as client:
        resp = await client.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": geo["lat"],
                "longitude": geo["lon"],
                "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
                "timezone": "auto",
                "forecast_days": 7
            }
        )
        data = resp.json()

    current = data["current"]
    daily = data["daily"]

    # Build a readable report
    report = f"Weather Report: {geo['name']}, {geo['country']}\n"
    report += "=" * 50 + "\n\n"
    report += f"Current Conditions:\n"
    report += f"  Temperature: {current['temperature_2m']}°C\n"
    report += f"  Humidity:    {current['relative_humidity_2m']}%\n"
    report += f"  Wind Speed:  {current['wind_speed_10m']} km/h\n\n"
    report += f"7-Day Forecast:\n"
    for i in range(7):
        report += (
            f"  {daily['time'][i]}: "
            f"{daily['temperature_2m_min'][i]}°C – {daily['temperature_2m_max'][i]}°C, "
            f"rain {daily['precipitation_sum'][i]} mm\n"
        )
    return report


# ── Tool 2: Get Country Info ─────────────────────────────────
# Fetches detailed country information from REST Countries API.

@mcp.tool()
async def get_country_info(country_name: str) -> str:
    """Get detailed information about a country (capital, population, languages, currency, region)."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"https://restcountries.com/v3.1/name/{country_name}",
            params={"fullText": "false"}
        )
        if resp.status_code != 200:
            return f"Country not found: {country_name}"
        countries = resp.json()

    c = countries[0]
    languages = ", ".join(c.get("languages", {}).values()) or "N/A"
    currencies = ", ".join(
        f"{v['name']} ({v.get('symbol', '')})" for v in c.get("currencies", {}).values()
    ) or "N/A"
    capitals = ", ".join(c.get("capital", [])) or "N/A"

    report = f"Country Profile: {c['name']['common']}\n"
    report += "=" * 50 + "\n\n"
    report += f"  Official Name: {c['name'].get('official', 'N/A')}\n"
    report += f"  Capital:       {capitals}\n"
    report += f"  Region:        {c.get('region', 'N/A')} / {c.get('subregion', 'N/A')}\n"
    report += f"  Population:    {c.get('population', 0):,}\n"
    report += f"  Languages:     {languages}\n"
    report += f"  Currencies:    {currencies}\n"
    report += f"  Timezone(s):   {', '.join(c.get('timezones', []))}\n"
    report += f"  Drives on:     {c.get('car', {}).get('side', 'N/A')} side\n"
    return report


# ── Tool 3: Get Public Holidays ─────────────────────────────
# Uses the Nager.Date API to list public holidays.

@mcp.tool()
async def get_holidays(country_code: str, year: int) -> str:
    """Get public holidays for a country. Requires a 2-letter ISO country code (e.g. 'JP' for Japan, 'FR' for France) and a year."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}"
        )
        if resp.status_code != 200:
            return f"Could not fetch holidays for {country_code} in {year}. Make sure the country code is a valid 2-letter ISO code."
        holidays = resp.json()

    report = f"Public Holidays: {country_code} ({year})\n"
    report += "=" * 50 + "\n\n"
    for h in holidays:
        report += f"  {h['date']}  {h['localName']} ({h['name']})\n"
    report += f"\nTotal: {len(holidays)} public holidays\n"
    return report


# ── Tool 4: Compare Weather ──────────────────────────────────
# Demonstrates internal tool chaining — calls _geocode and the
# weather API twice, then formats a side-by-side comparison.

@mcp.tool()
async def compare_weather(city_a: str, city_b: str) -> str:
    """Compare current weather between two cities side by side."""
    geo_a = await _geocode(city_a)
    geo_b = await _geocode(city_b)
    if not geo_a:
        return f"Could not find city: {city_a}"
    if not geo_b:
        return f"Could not find city: {city_b}"

    async with httpx.AsyncClient() as client:
        # Fetch both forecasts
        params_base = {
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
            "timezone": "auto"
        }
        resp_a = await client.get(
            "https://api.open-meteo.com/v1/forecast",
            params={**params_base, "latitude": geo_a["lat"], "longitude": geo_a["lon"]}
        )
        resp_b = await client.get(
            "https://api.open-meteo.com/v1/forecast",
            params={**params_base, "latitude": geo_b["lat"], "longitude": geo_b["lon"]}
        )
        wa = resp_a.json()["current"]
        wb = resp_b.json()["current"]

    # Side-by-side comparison
    header_a = f"{geo_a['name']}, {geo_a['country']}"
    header_b = f"{geo_b['name']}, {geo_b['country']}"
    report = f"Weather Comparison\n{'=' * 55}\n\n"
    report += f"{'Metric':<20} {header_a:<18} {header_b:<18}\n"
    report += f"{'-' * 56}\n"
    report += f"{'Temperature':<20} {wa['temperature_2m']}°C{'':<12} {wb['temperature_2m']}°C\n"
    report += f"{'Humidity':<20} {wa['relative_humidity_2m']}%{'':<13} {wb['relative_humidity_2m']}%\n"
    report += f"{'Wind Speed':<20} {wa['wind_speed_10m']} km/h{'':<9} {wb['wind_speed_10m']} km/h\n"

    # Verdict
    diff = wa['temperature_2m'] - wb['temperature_2m']
    if abs(diff) < 2:
        report += f"\nVerdict: Both cities have similar temperatures right now.\n"
    elif diff > 0:
        report += f"\nVerdict: {header_a} is {abs(diff):.1f}°C warmer than {header_b}.\n"
    else:
        report += f"\nVerdict: {header_b} is {abs(diff):.1f}°C warmer than {header_a}.\n"
    return report


# ── Resource 1: About ────────────────────────────────────────
# Resources are read-only reference data. The LLM doesn't call
# them as tools — instead the client reads them for context.

@mcp.resource("info://about")
def about() -> str:
    """About this MCP server."""
    return (
        "World Explorer MCP Server\n"
        "========================\n\n"
        "A travel research assistant that provides:\n"
        "  - Current weather and forecasts (Open-Meteo)\n"
        "  - Country profiles (REST Countries)\n"
        "  - Public holiday calendars (Nager.Date)\n\n"
        "All data comes from free, public APIs — no API keys required.\n"
    )


# ── Resource 2: Country Codes ────────────────────────────────
# A quick-reference list so the LLM knows which ISO codes to
# use when calling get_holidays().

@mcp.resource("data://country-codes")
def country_codes() -> str:
    """Common ISO 3166-1 alpha-2 country codes."""
    codes = [
        ("US", "United States"), ("GB", "United Kingdom"), ("FR", "France"),
        ("DE", "Germany"), ("JP", "Japan"), ("BR", "Brazil"),
        ("AU", "Australia"), ("IN", "India"), ("CN", "China"),
        ("CA", "Canada"), ("MX", "Mexico"), ("IT", "Italy"),
        ("ES", "Spain"), ("TH", "Thailand"), ("KR", "South Korea"),
        ("AR", "Argentina"), ("ZA", "South Africa"), ("EG", "Egypt"),
        ("NG", "Nigeria"), ("SE", "Sweden"), ("NO", "Norway"),
        ("NZ", "New Zealand"), ("PT", "Portugal"), ("TR", "Turkey"),
    ]
    result = "ISO Country Codes (for use with get_holidays)\n"
    result += "=" * 45 + "\n\n"
    for code, name in codes:
        result += f"  {code}  {name}\n"
    return result


# ── Prompt: Travel Brief ────────────────────────────────────
# Prompts are reusable templates that guide the LLM on how to
# combine multiple tools for a comprehensive response.

@mcp.prompt()
def travel_brief(destination: str) -> str:
    """Generate a comprehensive travel brief for a destination."""
    return (
        f"Create a comprehensive travel brief for {destination}. "
        f"Please gather and present the following information:\n\n"
        f"1. Use get_country_info to find the country details "
        f"(capital, languages, currency, population).\n"
        f"2. Use get_weather to check the current weather and forecast "
        f"for the capital or main city.\n"
        f"3. Use get_holidays to find upcoming public holidays "
        f"(use the current year).\n\n"
        f"Combine all findings into a well-organized travel brief with "
        f"practical tips for a visitor."
    )


# ── Entry point ──────────────────────────────────────────────
if __name__ == "__main__":
    mcp.run()

# 4. Understanding the Server

Let's walk through what we just wrote:

### Tools (4 total)
These are **actions the LLM can decide to call** during a conversation:

| Tool | What it does | APIs used |
|------|-------------|-----------|
| `get_weather(city)` | Geocodes the city, then fetches current weather + 7-day forecast | Open-Meteo Geocoding + Weather |
| `get_country_info(country_name)` | Returns capital, population, languages, currency, region | REST Countries |
| `get_holidays(country_code, year)` | Lists all public holidays for a country/year | Nager.Date |
| `compare_weather(city_a, city_b)` | Side-by-side weather comparison of two cities | Open-Meteo (called twice) |

Notice how `get_weather` and `compare_weather` both reuse the private `_geocode()` helper — this is **internal chaining** within the server.

### Resources (2 total)
These are **read-only reference data** the client can access:
- `info://about` — Describes what this server does
- `data://country-codes` — ISO code lookup table (useful when the LLM needs to call `get_holidays`)

### Prompt (1 total)
A **reusable prompt template** that instructs the LLM to chain multiple tools together:
- `travel_brief(destination)` — Guides the agent to gather country info + weather + holidays and combine them into a travel brief

# 5. Building the Client

## Configuring the LLM

In [ ]:
config = load_openai_config("the MCP World Explorer notebook")
OPENAI_API_KEY = config["OPENAI_API_KEY"]
OPENAI_API_BASE = config["OPENAI_API_BASE"]


In [ ]:
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
    model='gpt-4o-mini',
    temperature=0
)

## Server Parameters

The client will launch `server.py` as a subprocess and communicate via standard I/O (stdin/stdout). This is the MCP stdio transport — the same mechanism used by Claude Desktop, VS Code extensions, and other MCP hosts.

In [ ]:
server_params = StdioServerParameters(
    command=sys.executable,
    args=['server.py']
)


## Agent Function

The `run_agent` function:
1. Connects to the MCP server over stdio
2. Loads available tools from the server
3. Creates a ReAct agent (reason + act loop)
4. Sends the user query and prints the tool-call trace

A **ReAct agent** works by iteratively: thinking about what to do → calling a tool → observing the result → thinking again → until it has enough information to answer.

In [ ]:
async def run_agent(user_query):
    """Send a query to the World Explorer agent and print the tool-call trace."""
    print(f"Query: {user_query}")
    print("=" * 60)
    agent_response = None
    try:
        async with stdio_client(server_params, errlog=subprocess.PIPE) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                # Load tools from the MCP server
                tools = await load_mcp_tools(session)
                print(f"Available tools: {[t.name for t in tools]}")

                # Create a LangGraph ReAct agent with the MCP tools
                agent = create_react_agent(llm, tools)

                # Run the agent
                agent_response = await agent.ainvoke({
                    "messages": [HumanMessage(content=user_query)]
                })

                # Print tool call trace
                print("\nTool calls made:")
                for msg in agent_response["messages"]:
                    if hasattr(msg, 'tool_calls') and msg.tool_calls:
                        for tc in msg.tool_calls:
                            print(f"  -> {tc['name']}({tc['args']})")

        return agent_response
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()
        return {"error": str(e)}


# 6. Demo: Single Tool — Weather Lookup

The simplest case: the agent needs just one tool to answer the question.

In [ ]:
response = asyncio.run(run_agent("What's the weather like in Tokyo right now?"))

In [ ]:
print(response['messages'][-1].content)

# 7. Demo: Two-Tool Chain — Country + Weather

Here the agent must reason that it needs **two different tools** to fully answer the question — country info for languages/currency, and weather for Paris.

In [ ]:
response = asyncio.run(run_agent(
    "Tell me about France — what languages do they speak, what currency do they use, and what's the weather like in Paris?"
))

In [ ]:
print(response['messages'][-1].content)

# 8. Demo: Side-by-Side Comparison

The `compare_weather` tool handles this in a **single call** — but internally it makes two API requests (one per city). This shows how MCP tools can encapsulate complex logic.

In [ ]:
response = asyncio.run(run_agent("Compare the weather in Barcelona vs Bangkok"))

In [ ]:
print(response['messages'][-1].content)

# 9. Demo: Full Chain — Three APIs

This query requires the agent to call **three different tools** (country info, weather, and holidays), then synthesize the results into a coherent travel brief. Watch the tool-call trace to see the agent's reasoning process.

In [ ]:
response = asyncio.run(run_agent(
    "I'm planning a trip to Japan. Give me a complete travel brief — "
    "country info, Tokyo weather, and upcoming public holidays for 2026."
))

In [ ]:
print(response['messages'][-1].content)

# 10. Demo: Open-Ended Reasoning

This is the most interesting case. The agent must **decide on its own** which tools to call, what cities to check, and how to weigh the factors. There's no single "right" set of tool calls — the agent reasons about what information would be most useful for comparing beach holidays.

In [ ]:
response = asyncio.run(run_agent(
    "I'm deciding between Brazil and Thailand for a beach holiday. "
    "Compare both countries and their weather to help me decide."
))

In [ ]:
print(response['messages'][-1].content)

# 11. Conclusion

In this notebook, we built a **World Explorer** MCP server that demonstrates all three MCP primitives:

- **Tools** — The agent used `get_weather`, `get_country_info`, `get_holidays`, and `compare_weather` to fetch live data from three different public APIs
- **Resources** — `info://about` and `data://country-codes` provide static reference data that clients can read
- **Prompts** — `travel_brief(destination)` offers a reusable template for guiding multi-tool queries

### Key Takeaways

1. **The client stays simple.** We never wrote API-specific code on the client side — the MCP server handles all the HTTP calls, data formatting, and error handling.
2. **Tool chaining emerges naturally.** The LLM agent decides which tools to call and in what order based on the query, without any hardcoded orchestration logic.
3. **Internal composition is powerful.** The `compare_weather` tool and the shared `_geocode()` helper show how server-side tools can compose and reuse logic.
4. **Real APIs, zero keys.** All three APIs (Open-Meteo, REST Countries, Nager.Date) are free and require no authentication, making this notebook fully reproducible.

### Next Steps

- Add more tools: flight prices (Amadeus API), exchange rates (Open Exchange Rates), or points of interest (OpenTripMap)
- Try the SSE transport instead of stdio for remote server deployments
- Build a multi-server setup where the agent connects to several MCP servers simultaneously
- Explore MCP's sampling capability, where the server can request LLM completions from the client